In [13]:
!pip install tensorflow

In [14]:
import pandas as pd
import numpy as np
import tensorflow as tf

In [15]:
dataset = pd.read_csv('../data/Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values    # first three columns (0-2) are not useful to model
y = dataset.iloc[:, -1].values

In [16]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1,2])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [18]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_norm = sc.fit_transform(X_train)
X_test_norm = sc.transform(X_test)

In [19]:
ann = tf.keras.models.Sequential()
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [20]:
auc = tf.keras.metrics.AUC()
ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', auc])

ann.fit(X_train_norm, y_train, batch_size=32, epochs=100)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6009 - auc_2: 0.4978 - loss: 0.6647
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7997 - auc_2: 0.6084 - loss: 0.4951
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8032 - auc_2: 0.7331 - loss: 0.4429
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7979 - auc_2: 0.7677 - loss: 0.4348
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8038 - auc_2: 0.7726 - loss: 0.4282
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8107 - auc_2: 0.7961 - loss: 0.4159
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8209 - auc_2: 0.7983 - loss: 0.4073
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8299 - auc_2: 0.8142 - loss: 0.3923
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8349 - auc_2: 0.8269 - loss: 0.3823
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8474 

In [21]:
single_observation = [[ 600,"France","Male",40,3,60000,2,1,1,50000 ]]

X_single_obs = ct.transform(single_observation)
X_single_obs = sc.transform(X_single_obs)

ann.predict(X_single_obs)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step


array([[0.04515573]], dtype=float32)

In [22]:
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

y_pred_prob = ann.predict(X_test_norm)
y_pred_bin = (y_pred_prob > 0.5)

cm = confusion_matrix(y_true=y_test, y_pred=y_pred_bin)
acc = accuracy_score(y_true=y_test, y_pred=y_pred_bin)
auc = roc_auc_score(y_true=y_test, y_score=y_pred_prob)

print('Confusion Matrix:\n', cm)
print('Accuracy Score:', acc)
print('Area Under Curve:', auc)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 960us/step
Confusion Matrix:
 [[1523   72]
 [ 196  209]]
Accuracy Score: 0.866
Area Under Curve: 0.8678052556213476
